# Complexity-01 — Compter des pas

**Navigation** : [Index de la série](README.md) | [Version Python](Complexity-01-StepCounting-Python.ipynb) | [Complexity-02 — Vérifier ou trouver >>](Complexity-02-P-NP-Reduction-Python.ipynb)


> **Environnement** : C# et .NET 9+; noyau .NET Interactive. Le notebook utilise seulement la BCL.

> **Public : Découverte.** Premier notebook de la série `Complexity/`. Il ne suppose aucune
> notion de théorie de la complexité : tout ce qu'il utilise est construit ici, cellule par cellule.

## Ce que ce notebook suppose

- lire un programme C# simple : boucles `for` et `while`, fonctions, tableaux et collections C# ;
- savoir ce que font un tri et une recherche dans une collection.

Rien d'autre : ni notation $O$, ni machine de Turing, ni classe de complexité.

## Ce que vous saurez faire à la fin

1. expliquer pourquoi on mesure le coût d'un algorithme en **nombre de pas** plutôt qu'en secondes ;
2. définir la **taille d'une entrée** et la **fonction de coût** d'un algorithme ;
3. faire tourner une petite machine de Turing et compter ses pas ;
4. reconnaître à la mesure trois croissances, $n \log n$, $n^2$ et $2^n$, grâce au **test du doublement** ;
5. dire, pour chacune, quelle taille d'entrée reste traitable en une seconde, une minute, un jour.

**Durée estimée** : 45 minutes.


In [1]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.Globalization;
using System.Linq;
using System.Numerics;
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.GetCultureInfo("fr-FR");
Console.WriteLine($"Kernel C# actif : .NET {Environment.Version}; expérience CPU seule, BCL uniquement.");

Kernel C# actif : .NET 10.0.14; expérience CPU seule, BCL uniquement.


## 1. Le chronomètre ment un peu

Le réflexe naturel pour savoir si un programme est rapide, c'est de le chronométrer. Faisons-le :
la même fonction, sur la même liste, exécutée sept fois de suite.

La fonction `Somme` additionne un million d'entiers. Elle tient aussi un compteur `pas`,
augmenté de 1 à chaque addition : c'est la mesure que ce notebook va opposer au chronomètre.
Si le temps mesuré était une propriété du programme, les sept essais devraient afficher le
même nombre de millisecondes. Regardez ce qui varie d'un essai à l'autre, et ce qui ne varie pas.

In [2]:
long Somme(List<int> liste, out long pas)
{
    long total = 0;
    pas = 0;
    foreach (int x in liste) { total += x; pas++; }
    return total;
}
var donneesSomme = Enumerable.Range(0, 1_000_000).ToList();
var chronosSomme = new List<double>();
long sommeTotal = 0, pasSomme = 0;
for (int essai = 0; essai < 7; essai++)
{
    var chrono = Stopwatch.StartNew();
    sommeTotal = Somme(donneesSomme, out pasSomme);
    chrono.Stop();
    chronosSomme.Add(chrono.Elapsed.TotalMilliseconds);
}
for (int essai = 0; essai < chronosSomme.Count; essai++)
    Console.WriteLine($"essai {essai + 1} : {chronosSomme[essai],7:F2} ms | {pasSomme:N0} pas");
if(sommeTotal!=499_999_500_000L || pasSomme!=1_000_000L || chronosSomme.Count!=7)throw new Exception("Somme/compteur exacts inattendus.");
Console.WriteLine($"somme = {sommeTotal:N0}; écart chrono lent/rapide = x{chronosSomme.Max()/chronosSomme.Min():F2}; pas identiques aux 7 essais = {pasSomme == 1_000_000}");

essai 1 :    8,27 ms | 1 000 000 pas


essai 2 :    7,46 ms | 1 000 000 pas


essai 3 :   10,19 ms | 1 000 000 pas


essai 4 :    6,78 ms | 1 000 000 pas


essai 5 :    7,89 ms | 1 000 000 pas


essai 6 :    6,55 ms | 1 000 000 pas


essai 7 :    7,09 ms | 1 000 000 pas


somme = 499 999 500 000; écart chrono lent/rapide = x1,56; pas identiques aux 7 essais = True


**Lecture.** Les sept chronos ne coïncident pas : l'écart affiché dépend de ce que fait la
machine au même moment (autres programmes, fréquence du processeur, mémoire cache). Sur un autre
ordinateur, ou dans un autre langage, ils seraient tous différents. Le **nombre de pas**, lui, est
le même aux sept essais — et il serait le même sur n'importe quelle machine : c'est une
propriété de l'**algorithme**, pas de l'ordinateur.

C'est tout le point de départ de la théorie de la complexité : on ne demande pas « combien de
secondes ? » mais « **combien de pas** ? ». Le chronomètre reste utile pour confirmer une mesure ;
il ne sert pas à la définir.

## 2. Taille d'entrée et fonction de coût

Un algorithme ne coûte pas « 1000 pas » dans l'absolu : son coût dépend de l'entrée qu'on lui
donne. On résume une entrée par sa **taille** $n$ — le nombre d'éléments d'une liste, le nombre de
caractères d'un mot — et on étudie comment le nombre de pas évolue quand $n$ grandit.

Premier exemple : chercher une valeur dans une liste non triée en la parcourant du début à la fin.
Un pas = une comparaison.

In [3]:
(int index, int comparisons) RechercheLineaire(IReadOnlyList<int> liste, int valeur)
{
    int comparaisons = 0;
    for (int i = 0; i < liste.Count; i++) { comparaisons++; if (liste[i] == valeur) return (i, comparaisons); }
    return (-1, comparaisons);
}
Console.WriteLine($"{"n",7} | {"meilleur cas",12} | {"pire cas",9} | pire cas / n");
Console.WriteLine(new string('-', 48));
foreach (int n in new[]{10,100,1_000,10_000,100_000})
{
    var liste = Enumerable.Range(0,n).ToArray();
    int meilleur = RechercheLineaire(liste,0).comparisons;
    int pire = RechercheLineaire(liste,-1).comparisons;
    if(meilleur!=1 || pire!=n)throw new Exception($"Recherche linéaire incorrecte à n={n}.");
    Console.WriteLine($"{n,7:N0} | {meilleur,12} | {pire,9:N0} | {(double)pire/n,11:F2}");
}

      n | meilleur cas |  pire cas | pire cas / n


------------------------------------------------


     10 |            1 |        10 |        1,00


    100 |            1 |       100 |        1,00


  1 000 |            1 |     1 000 |        1,00


 10 000 |            1 |    10 000 |        1,00


100 000 |            1 |   100 000 |        1,00


**Lecture.** Pour une même taille $n$, le coût varie selon l'entrée : **1** comparaison si la
valeur est en tête, **$n$** si elle est absente. On retient en général le **pire cas** : la
**fonction de coût** $T(n)$ est le plus grand nombre de pas que l'algorithme peut faire sur une
entrée de taille $n$. Ici la dernière colonne vaut exactement 1 : $T(n) = n$. La recherche
linéaire a un coût **linéaire** — doubler la liste double le travail dans le pire cas.

Ce vocabulaire suffit pour la suite : **taille** $n$, **pas**, **pire cas**, **fonction de coût** $T(n)$.

## Exercices

Les trois exercices suivent l'ordre du notebook. Chaque cellule s'exécute sans erreur telle
qu'elle est livrée : remplacez les lignes `# TODO etudiant` par votre solution.

### Exercice 1 — Compter les comparaisons de la recherche dichotomique

Dans une liste **triée**, on peut chercher une valeur bien plus vite qu'en section 2 : on compare
avec l'élément du milieu, puis on ne garde que la moitié où la valeur peut se trouver.

- **Etape 1 :** affecter une fonction à `rechercheDichotomique` qui renvoie le couple
  `(indice ou -1, nombre de comparaisons)` — une comparaison = un test de `x` contre un élément.
  Pour les fixtures, utiliser l’intervalle fermé `[bas, haut]` et le milieu arrondi vers le bas.
- **Etape 2 :** mesurer les deux absences, `x=-1` (sous le minimum) et `x=n` (au-dessus du maximum),
  pour $n = 2^k$, $k = 4, 8, 12, 16, 20$; vérifier l’indice `-1` et compter les comparaisons.
  Comparer ces comptes au coût linéaire.
- **Etape 3 :** appliquer le test du doublement. Quelle croissance reconnaissez-vous ?

In [4]:
Func<IReadOnlyList<int>,int,(int index,int comparisons)> rechercheDichotomique = null!; // TODO etudiant : implémenter la recherche et compter les comparaisons
if(rechercheDichotomique is null)Console.WriteLine("Exercice à compléter : écrire la recherche dichotomique et son compteur.");
else
{
    foreach(var (liste,x,idx,compteAttendu) in new[]{(new[]{0,3,5,8,11,14,19},11,4,3),(new[]{0,3,5,8,11,14,19},7,-1,3),(new[]{2},2,0,1),(Array.Empty<int>(),9,-1,0)})
    {
        var obtenu=rechercheDichotomique(liste,x);
        if(obtenu.index!=idx || obtenu.comparisons!=compteAttendu)throw new Exception("Exercice 1 : indice ou compteur incorrect sur une fixture discriminante.");
    }
    foreach(int k in new[]{4,8,12,16,20})
    {
        int n=1<<k;var liste=Enumerable.Range(0,n).ToArray();
        var (indiceBas,compteBas)=rechercheDichotomique(liste,-1);
        var (indiceHaut,compteHaut)=rechercheDichotomique(liste,n);
        if(indiceBas!=-1 || compteBas!=k || indiceHaut!=-1 || compteHaut!=k+1)throw new Exception($"Exercice 1 : les recherches absentes doivent coûter {k} comparaisons en dessous et {k+1} au-dessus.");
        Console.WriteLine($"n=2^{k,-2}: absent bas {compteBas}, absent haut {compteHaut}; recherche linéaire {n:N0}.");
    }
}

Exercice à compléter : écrire la recherche dichotomique et son compteur.


## 3. Une machine qui compte ses pas : la machine de Turing

« Un pas » d'un programme C# reste un peu flou : une addition, un appel de fonction, une
lecture de liste coûtent-ils la même chose ? Pour rendre la notion **exacte**, les informaticiens
utilisent un modèle de machine où le pas est défini sans ambiguïté : la **machine de Turing**
(Alan Turing, 1936).

- un **ruban** infini découpé en cases, chaque case porte un symbole (`_` = case vide) ;
- une **tête** de lecture posée sur une case ;
- un **état** courant, pris dans une liste finie ;
- une **table de transitions** : pour chaque couple (état, symbole lu), elle dit quel symbole
  écrire, dans quel sens déplacer la tête (`L` gauche, `R` droite, `S` rester) et dans quel état passer.

**Un pas = une transition.** La machine s'arrête quand elle atteint un état acceptant, ou quand
aucune transition ne s'applique (elle rejette). Le simulateur ci-dessous accepte plusieurs rubans
(on n'en utilisera qu'un ici) et compte ses pas.

Première machine : ajouter 1 à un nombre écrit en binaire. Elle va au bout du mot, puis revient
vers la gauche en propageant la retenue.

In [5]:
#nullable enable
class Ruban
{
    public Dictionary<int,char> Cellules { get; } = new();
    public int Tete { get; set; }
    public Ruban(string mot = "") { for (int i=0;i<mot.Length;i++) Cellules[i]=mot[i]; }
    public char Lire() => Cellules.TryGetValue(Tete,out char c) ? c : '_';
    public void Ecrire(char c) => Cellules[Tete]=c;
    public void Deplacer(char d) => Tete += d=='R' ? 1 : d=='L' ? -1 : 0;
}
record ActionRuban(char Ecrire, char Direction);
record RegleTuring(string NouvelEtat, ActionRuban[] Actions);
class MachineTuring
{
    public int K { get; }
    public Dictionary<string,RegleTuring> Delta { get; }
    public string Initial { get; }
    public HashSet<string> Acceptants { get; }
    public long Pas { get; private set; }
    public MachineTuring(int k, Dictionary<string,RegleTuring> delta, string initial, IEnumerable<string> acceptants)
    { K=k; Delta=delta; Initial=initial; Acceptants=new HashSet<string>(acceptants); }
    public (string? Etat, List<Ruban> Rubans) Run(string mot, long budget=100_000)
    {
        var rubans=Enumerable.Range(0,K).Select(i=>new Ruban(i==0?mot:"")).ToList();
        string etat=Initial; Pas=0;
        while(!Acceptants.Contains(etat))
        {
            if(Pas>=budget) return (null,rubans);
            string cle=Cle(etat,rubans.Select(r=>r.Lire()));
            if(!Delta.TryGetValue(cle,out var regle)) return (null,rubans);
            etat=regle.NouvelEtat;
            for(int i=0;i<K;i++){rubans[i].Ecrire(regle.Actions[i].Ecrire);rubans[i].Deplacer(regle.Actions[i].Direction);}
            Pas++;
        }
        return (etat,rubans);
    }
    public static string Cle(string etat,IEnumerable<char> symboles) => etat+"|"+string.Join("|",symboles);
}
string LireRuban(Ruban r)
{
    if(r.Cellules.Count==0)return "";
    int debut=r.Cellules.Keys.Min(),fin=r.Cellules.Keys.Max();
    return new string(Enumerable.Range(debut,fin-debut+1).Select(i=>r.Cellules.TryGetValue(i,out char c)?c:'_').ToArray()).Trim('_');
}
var transitionsIncrement = new Dictionary<string,RegleTuring>();
void AjouteIncrement(string etat,string lu,string nouvel, char ecrit,char dir) => transitionsIncrement[MachineTuring.Cle(etat,lu)] = new RegleTuring(nouvel,new[]{new ActionRuban(ecrit,dir)});
AjouteIncrement("q0","0","q0",'0','R'); AjouteIncrement("q0","1","q0",'1','R'); AjouteIncrement("q0","_","q1",'_','L');
AjouteIncrement("q1","1","q1",'0','L'); AjouteIncrement("q1","0","qF",'1','S'); AjouteIncrement("q1","_","qF",'1','S');
var increment = new MachineTuring(1,transitionsIncrement,"q0",new[]{"qF"});
foreach(var (mot,attendu) in new[]{("1011","1100"),("0","1"),("111","1000")})
{
    var (etat,rubans)=increment.Run(mot); string resultat=LireRuban(rubans[0]);
    if(resultat!=attendu || etat!="qF") throw new Exception($"Incrément incorrect {mot}->{resultat}");
    Console.WriteLine($"increment({mot,7}) -> {resultat,7} en {increment.Pas} pas");
}
int incrementChecks=0;
for(int longueur=1;longueur<=8;longueur++)
{
    for(int masque=0;masque<(1<<longueur);masque++)
    {
        string mot=new string(Enumerable.Range(0,longueur).Select(i=>((masque>>(longueur-i-1))&1)==0?'0':'1').ToArray());
        string valeur=Convert.ToString(Convert.ToInt32(mot,2)+1,2);
        string attendu=new string('0',Math.Max(0,mot.Length-valeur.Length))+valeur;
        int train=mot.Reverse().TakeWhile(c=>c=='1').Count();
        var (etat,rubans)=increment.Run(mot);
        if(etat!="qF" || LireRuban(rubans[0])!=attendu || increment.Pas!=mot.Length+train+2)throw new Exception($"Oracle incrément indépendant en désaccord sur {mot}.");
        incrementChecks++;
    }
}
Console.WriteLine($"Oracle incrément indépendant : {incrementChecks} mots binaires (longueurs 1–8), résultat et n+k+2 vérifiés.");

increment(   1011) ->    1100 en 8 pas


increment(      0) ->       1 en 3 pas


increment(    111) ->    1000 en 8 pas


Oracle incrément indépendant : 510 mots binaires (longueurs 1–8), résultat et n+k+2 vérifiés.


**Lecture.** La machine donne le bon résultat sur les trois mots (le contrôle le vérifie) et le
nombre de pas est **exact** : pour `'1011'`, 5 pas pour aller au bout du mot (4 chiffres, puis la
case vide) et 3 pas pour revenir en propageant la retenue. Aucun chronomètre n'intervient.

Deuxième machine, plus coûteuse : reconnaître un **palindrome** (un mot qui se lit pareil dans les
deux sens, comme `0110`). Avec une seule tête, la machine fait des allers-retours : elle raye le
premier symbole, court jusqu'au dernier, vérifie qu'il est identique, le raye, revient au début,
et recommence.

In [6]:
#nullable enable
var transitionsPal = new Dictionary<string,RegleTuring>();
void AjoutePal(string etat,string lu,string nouvel,ActionRuban action) => transitionsPal[MachineTuring.Cle(etat,lu)] = new RegleTuring(nouvel,new[]{action});
AjoutePal("scan","0","marque0",new ActionRuban('x','R')); AjoutePal("scan","1","marque1",new ActionRuban('x','R')); AjoutePal("scan","x","scan",new ActionRuban('x','R')); AjoutePal("scan","_","fin",new ActionRuban('_','S'));
foreach(string s in new[]{"0","1"}) { AjoutePal("marque0",s,"marque0",new ActionRuban(s[0],'R')); AjoutePal("marque1",s,"marque1",new ActionRuban(s[0],'R')); }
AjoutePal("marque0","x","verif0",new ActionRuban('x','L')); AjoutePal("marque0","_","verif0",new ActionRuban('_','L')); AjoutePal("marque1","x","verif1",new ActionRuban('x','L')); AjoutePal("marque1","_","verif1",new ActionRuban('_','L'));
AjoutePal("verif0","0","retour",new ActionRuban('x','L')); AjoutePal("verif1","1","retour",new ActionRuban('x','L')); AjoutePal("verif0","1","rejeter",new ActionRuban('1','S')); AjoutePal("verif1","0","rejeter",new ActionRuban('0','S')); AjoutePal("verif0","x","fin",new ActionRuban('x','S')); AjoutePal("verif1","x","fin",new ActionRuban('x','S'));
foreach(string s in new[]{"0","1","x"}) AjoutePal("retour",s,"retour",new ActionRuban(s[0],'L')); AjoutePal("retour","_","scan",new ActionRuban('_','R'));
var palindrome = new MachineTuring(1,transitionsPal,"scan",new[]{"fin"});
foreach(var (mot,attendu) in new[]{("0110","fin"),("0100",(string?)null),("0","fin"),("0110110","fin")})
{
    var (etat,_) = palindrome.Run(mot);
    if(etat!=attendu) throw new Exception($"Test palindrome incorrect pour {mot}");
}
long OraclePasPalindrome(string mot)
{
    int n=mot.Length, j=0; long pas=0;
    if(n==0)return 1;
    while(n-2*j>=2)
    {
        int restant=n-2*j;
        pas += j + 1 + (restant-1) + 1 + 1; // scan des marques, rature, balayage, blanc, comparaison
        if(mot[j]!=mot[n-j-1])return pas; // rejet au premier couple discordant
        pas += n-j; // retour vers le bord gauche et reprise du scan
        j++;
    }
    return pas + (n-2*j==1 ? j+3 : n+1); // centre impair ou balayage final des marques
}
int palindromeChecks=0;
for(int longueur=0;longueur<=8;longueur++)
{
    for(int masque=0;masque<(1<<longueur);masque++)
    {
        string mot=new string(Enumerable.Range(0,longueur).Select(i=>((masque>>(longueur-i-1))&1)==0?'0':'1').ToArray());
        bool attendu=mot.SequenceEqual(mot.Reverse());
        var (etat,_)=palindrome.Run(mot,500_000);
        if((etat=="fin")!=attendu || palindrome.Pas!=OraclePasPalindrome(mot))throw new Exception($"Oracle palindrome indépendant en désaccord sur {mot}.");
        palindromeChecks++;
    }
}
Console.WriteLine($"Oracle palindrome indépendant : {palindromeChecks} mots de longueur 0–8, verdict et transitions vérifiés.");
Console.WriteLine();
Console.WriteLine($"{"n",4} | {"pas inc",8} | {"pas pal",8} | {"pal / inc",9} | {"pal / n²",9}");
Console.WriteLine(new string('-',52));
foreach(int demi in new[]{4,8,16,32,64})
{
    string moitie=new string(Enumerable.Range(0,demi).Select(i=>(i*7)%2==0?'0':'1').ToArray());
    string mot=moitie+new string(moitie.Reverse().ToArray()); int n=mot.Length;
    increment.Run(mot); palindrome.Run(mot,500_000);
    Console.WriteLine($"{n,4} | {increment.Pas,8} | {palindrome.Pas,8} | {(double)palindrome.Pas/increment.Pas,8:F2}x | {(double)palindrome.Pas/(n*n),9:F3}");
}

Oracle palindrome indépendant : 511 mots de longueur 0–8, verdict et transitions vérifiés.


   n |  pas inc |  pas pal | pal / inc |  pal / n²


----------------------------------------------------


   8 |       10 |       69 |     6,90x |     1,078


  16 |       18 |      233 |    12,94x |     0,910


  32 |       34 |      849 |    24,97x |     0,829


  64 |       66 |     3233 |    48,98x |     0,789


 128 |      130 |    12609 |    96,99x |     0,770


**Lecture.** Sur les mêmes mots, les deux machines ne vivent pas dans le même monde :

- la machine d'incrément fait $n + 2$ pas sur les mots de la colonne `pas inc` : un aller, et une
  retenue qui s'éteint tout de suite parce que ces mots finissent par 0. Ce n'est pas le pire cas —
  le carnet l'a déjà mesuré : `increment.Run("111")`, un mot de tous les 1, coûte 8 pas pour $n = 3$,
  la retenue courant sur tout le mot. Son coût reste **linéaire** ;
- la machine palindrome fait de l'ordre de $n^2$ pas : la dernière colonne, `pal / n^2`, décroît
  de moins en moins vite (l'écart d'une ligne à la suivante est à peu près divisé par deux) et
  converge vers une constante. Son coût est **quadratique** : chaque symbole rayé coûte un
  aller-retour sur tout le mot.

Conséquence mesurée dans la colonne `pal / inc` : l'écart entre les deux machines **double** à
chaque doublement de $n$. Ce n'est pas une question de constante ou de machine plus rapide : c'est
la **forme** de la croissance qui diffère. C'est cette forme que la théorie de la complexité étudie.

### Exercice 2 — Écrire une machine de Turing et prédire ses pas

Construisez une machine à un ruban qui **accepte** les mots sur $\{0, 1\}$ contenant un nombre
**pair** de `1` (par exemple `0110`, `1001`, `000`, le mot vide) et **rejette** les autres
(`1`, `0111`).

- **Indice :** deux états suffisent pour lire le mot, `pair` et `impair`, plus un état acceptant
  atteint quand la tête lit la case vide `_` dans l'état `pair`. Ne mettez **aucune** transition
  sur `_` depuis l'état `impair` : l'absence de transition vaut rejet.
- **Etape 1 :** renseigner `transitionsParite` : chaque clé est `état|symbole` et chaque valeur contient le nouvel état et une action `ActionRuban`.
- **Etape 2 :** avant d'exécuter, **prédire** le nombre de pas sur un mot accepté de longueur $n$.
- **Etape 3 :** vérifier la prédiction pour $n = 10, 100, 1000$.

In [7]:
var transitionsParite = new Dictionary<string,RegleTuring>(); // TODO etudiant : table de parité des 1
if(transitionsParite.Count==0)Console.WriteLine("Exercice à compléter : la table de transitions pour la parité est vide.");
else
{
    var clesAttendues=new HashSet<string>(new[]{MachineTuring.Cle("pair","0"),MachineTuring.Cle("pair","1"),MachineTuring.Cle("pair","_"),MachineTuring.Cle("impair","0"),MachineTuring.Cle("impair","1")});
    if(transitionsParite.Count!=5 || !transitionsParite.Keys.ToHashSet().SetEquals(clesAttendues))throw new Exception("Exercice 2 : la table doit avoir exactement les cinq transitions de parité requises.");
    var parite=new MachineTuring(1,transitionsParite,"pair",new[]{"accepte"});
    var mots=new List<string>{""};
    for(int longueur=1;longueur<=8;longueur++)foreach(int masque in Enumerable.Range(0,1<<longueur))mots.Add(new string(Enumerable.Range(0,longueur).Select(i=>((masque>>i)&1)==0?'0':'1').ToArray()));
    foreach(string mot in mots){var (etat,_)=parite.Run(mot,1000);bool pair=mot.Count(c=>c=='1')%2==0;if((etat=="accepte")!=pair)throw new Exception($"Exercice 2 : parité incorrecte sur {mot}.");if(parite.Pas!=mot.Length+(pair?1:0))throw new Exception($"Exercice 2 : compteur inattendu sur {mot}.");}
    foreach(int n in new[]{10,100,1000})
    {
        string mot=new string('0',n-2)+"11";
        var (etat,_)=parite.Run(mot,2000);
        if(etat!="accepte" || parite.Pas!=n+1)throw new Exception($"Exercice 2 : prédiction n+1 incorrecte pour n={n}.");
        Console.WriteLine($"Mot accepté de longueur {n} : {parite.Pas} transitions, comme prédit.");
    }
    Console.WriteLine($"Exercice 2 vérifié exhaustivement : {mots.Count} mots binaires de longueur 0 à 8; accepte n+1, rejette n transitions.");
}

Exercice à compléter : la table de transitions pour la parité est vide.


## 4. Trois croissances, un test : doubler $n$

Comptons les pas nous-mêmes sur trois algorithmes aux comportements très
différents :

| Algorithme | Ce qu'il fait | Un pas = |
|---|---|---|
| **tri fusion** | coupe la liste en deux, trie chaque moitié, fusionne | une comparaison |
| **paire la plus proche** | compare toutes les paires de nombres | une comparaison |
| **tous les sous-ensembles** | examine chaque sous-ensemble d'une liste de poids et calcule sa somme | un sous-ensemble examiné |

Le **test du doublement** consiste à mesurer $T(2n) / T(n)$ : combien de fois plus de travail
quand l'entrée double ?

Les séquences sont produites par `System.Random` avec la graine 0 : elles diffèrent de NumPy côté Python, mais rendent ces comptes reproductibles dans ce jumeau.

In [8]:
long TriFusion(int[] liste,ref long compteur)
{
    if(liste.Length<=1)return 0;
    int m=liste.Length/2; int[] g=liste[..m], d=liste[m..]; TriFusion(g,ref compteur); TriFusion(d,ref compteur);
    int i=0,j=0,k=0;
    while(i<g.Length && j<d.Length){compteur++;liste[k++]=g[i]<=d[j]?g[i++]:d[j++];}
    while(i<g.Length)liste[k++]=g[i++]; while(j<d.Length)liste[k++]=d[j++]; return compteur;
}
long PairesProches(int[] liste)
{
    long compteur=0; long meilleur=long.MaxValue;
    for(int i=0;i<liste.Length;i++)for(int j=i+1;j<liste.Length;j++){compteur++;meilleur=Math.Min(meilleur,Math.Abs((long)liste[i]-liste[j]));}
    return compteur;
}
long SommesSousEnsembles(int[] poids)
{
    int nombre=1<<poids.Length; var sommes=new long[nombre]; long compteur=0;
    for(int masque=1;masque<nombre;masque++){int bas=masque & -masque; sommes[masque]=sommes[masque^bas]+poids[System.Numerics.BitOperations.TrailingZeroCount((uint)bas)]; compteur++;}
    return compteur+1; // le sous-ensemble vide
}
int[] Donnees(int n,int graine)
{
    var rng=new Random(graine); var a=new int[n]; for(int i=0;i<n;i++)a[i]=rng.Next(0,1_000_000); return a;
}
long CompterTri(int n){var a=Donnees(n,0);long c=0;TriFusion(a,ref c);if(!a.SequenceEqual(a.OrderBy(x=>x)))throw new Exception("Tri fusion incorrect");return c;}
long CompterPaires(int n){long compte=PairesProches(Donnees(n,0));long attendu=(long)n*(n-1)/2;if(compte!=attendu)throw new Exception($"Comptage exhaustif des paires incorrect à n={n}");return compte;}
long CompterSous(int n)=>SommesSousEnsembles(Donnees(n,0));
void TestDoublement(string nom,Func<int,long> algo,int[] tailles)
{
    Console.WriteLine(nom);
    long precedent=0;
    for(int k=0;k<tailles.Length;k++){long pas=algo(tailles[k]);string rapport=k==0?"":$"x{(double)pas/precedent:F2}";Console.WriteLine($"   n = {tailles[k],6:N0} : {pas,12:N0} pas {rapport}");precedent=pas;}
}
TestDoublement("Tri fusion (n double à chaque ligne)",CompterTri,new[]{1000,2000,4000,8000,16000});
TestDoublement("Paire la plus proche (n double à chaque ligne)",CompterPaires,new[]{250,500,1000,2000});
TestDoublement("Tous les sous-ensembles (n double à chaque ligne)",CompterSous,new[]{5,10,20});
TestDoublement("Tous les sous-ensembles (n augmente de 1)",CompterSous,new[]{16,17,18,19,20});

Tri fusion (n double à chaque ligne)


   n =  1 000 :        8 716 pas 


   n =  2 000 :       19 416 pas x2,23


   n =  4 000 :       42 810 pas x2,20


   n =  8 000 :       93 625 pas x2,19


   n = 16 000 :      203 252 pas x2,17


... (11 outputs left out)

   n =     17 :      131 072 pas x2,00


   n =     18 :      262 144 pas x2,00


   n =     19 :      524 288 pas x2,00


   n =     20 :    1 048 576 pas x2,00


**Lecture.** Le test du doublement sépare nettement les trois algorithmes :

- **tri fusion** : quand $n$ double, le travail fait un peu plus que doubler (rapports affichés
  légèrement au-dessus de 2). C'est la signature de $n \log n$ : $2n \log(2n) / (n \log n) = 2 \cdot (1 + 1/\log_2 n)$ ;
- **paire la plus proche** : le nombre de comparaisons est multiplié par $4 + 2/(n-1)$ quand $n$ double : il est proche de 4, sans lui être égal. La formule vient de $n(n-1)/2$ paires et sa croissance est quadratique ;
- **tous les sous-ensembles** : le test du doublement n'a plus de sens, le rapport explose
  (×32 puis ×1024). La bonne question devient « que se passe-t-il quand $n$ augmente de **un** ? »
  — et la dernière table répond : le travail **double** à chaque élément ajouté. C'est la
  signature de $2^n$.

Un graphique en échelles logarithmiques rend la différence visible d'un coup d'œil : un coût
polynomial $n^k$ y devient une **droite de pente $k$**, alors qu'une exponentielle n'est une
droite d'aucune pente.

In [9]:
var taillesTri = Enumerable.Range(4,11).Select(k=>1<<k).ToArray();
var taillesPaires = Enumerable.Range(3,9).Select(k=>1<<k).ToArray();
var taillesSous = Enumerable.Range(2,19).ToArray();
var courbeTri = taillesTri.Select(CompterTri).ToArray();
var courbePaires = taillesPaires.Select(CompterPaires).ToArray();
var courbeSous = taillesSous.Select(CompterSous).ToArray();
double PenteLogLog(int[] xs,long[] ys)
{
    var lx=xs.Select(x=>Math.Log(x)).ToArray(); var ly=ys.Select(y=>Math.Log(y)).ToArray();
    double mx=lx.Average(),my=ly.Average(); return lx.Zip(ly,(x,y)=>(x-mx)*(y-my)).Sum()/lx.Sum(x=>(x-mx)*(x-mx));
}
string[] couleurs={"#1769aa","#c2410c","#16803c"};
int gauche=94,haut=48,largeur=500,hauteur=390;
double minX=Math.Log10(2),maxX=Math.Log10(20000),minY=0,maxY=Math.Log10(2_000_000);
double X(double x)=>gauche+(Math.Log10(x)-minX)/(maxX-minX)*largeur;
double Y(double y)=>haut+hauteur-(Math.Log10(Math.Max(1,y))-minY)/(maxY-minY)*hauteur;
var svg=new System.Text.StringBuilder();
svg.Append($"<svg xmlns='http://www.w3.org/2000/svg' width='850' height='540' viewBox='0 0 850 540'><rect width='850' height='540' fill='#ffffff'/><text x='425' y='27' text-anchor='middle' font-family='sans-serif' font-size='17'>Trois croissances comptées pas à pas</text>");
foreach(int tick in new[]{10,100,1000,10000}){double x=X(tick);svg.Append(FormattableString.Invariant($"<line x1='{x:F2}' y1='{haut}' x2='{x:F2}' y2='{haut+hauteur}' stroke='#dddddd'/><text x='{x:F2}' y='{haut+hauteur+22}' text-anchor='middle' font-family='sans-serif' font-size='12'>10^{Math.Log10(tick):0}</text>"));}
foreach(int tick in new[]{1,10,100,1000,10000,100000,1000000}){double y=Y(tick);svg.Append(FormattableString.Invariant($"<line x1='{gauche}' y1='{y:F2}' x2='{gauche+largeur}' y2='{y:F2}' stroke='#dddddd'/><text x='{gauche-10}' y='{y+4:F2}' text-anchor='end' font-family='sans-serif' font-size='12'>{tick:N0}</text>"));}
svg.Append(FormattableString.Invariant($"<line x1='{gauche}' y1='{haut}' x2='{gauche}' y2='{haut+hauteur}' stroke='#222' stroke-width='1.5'/><line x1='{gauche}' y1='{haut+hauteur}' x2='{gauche+largeur}' y2='{haut+hauteur}' stroke='#222' stroke-width='1.5'/><text x='{gauche+largeur/2}' y='{haut+hauteur+48}' text-anchor='middle' font-family='sans-serif' font-size='13'>taille de l’entrée n (échelle log)</text><text x='20' y='{haut+hauteur/2}' transform='rotate(-90 20 {haut+hauteur/2})' text-anchor='middle' font-family='sans-serif' font-size='13'>nombre de pas (échelle log)</text>"));
var series=new[]{(taillesTri,courbeTri,"Tri fusion : n log n"),(taillesPaires,courbePaires,"Paires : n²"),(taillesSous,courbeSous,"Sous-ensembles : 2^n")};
for(int s=0;s<series.Length;s++)
{
    var (xs,ys,label)=series[s];string pts=string.Join(" ",xs.Select((x,i)=>FormattableString.Invariant($"{X(x):F2},{Y(ys[i]):F2}")));
    svg.Append($"<polyline points='{pts}' fill='none' stroke='{couleurs[s]}' stroke-width='2.5'/>");
    for(int i=0;i<xs.Length;i++)svg.Append(FormattableString.Invariant($"<circle cx='{X(xs[i]):F2}' cy='{Y(ys[i]):F2}' r='3.3' fill='{couleurs[s]}'/>"));
    int legY=62+s*22;svg.Append($"<line x1='620' y1='{legY}' x2='644' y2='{legY}' stroke='{couleurs[s]}' stroke-width='3'/><text x='650' y='{legY+4}' font-family='sans-serif' font-size='12'>{System.Security.SecurityElement.Escape(label)}</text>");
}
svg.Append("</svg>");
record PlotSvg(string Content);
Microsoft.DotNet.Interactive.Formatting.Formatter.Register<PlotSvg>((plot,writer)=>writer.Write(plot.Content),"text/html");
display(new PlotSvg(svg.ToString()));
Console.WriteLine($"Pentes log-log : tri fusion {PenteLogLog(taillesTri,courbeTri):F2}, paires {PenteLogLog(taillesPaires,courbePaires):F2}.");
var pentesLocales=Enumerable.Range(0,6).Select(k=>{int j=3*k;return Math.Log((double)courbeSous[j+1]/courbeSous[j])/Math.Log((double)taillesSous[j+1]/taillesSous[j]);});
Console.WriteLine("Pentes locales des sous-ensembles (échantillon) : "+string.Join("  ",pentesLocales.Select(x=>x.ToString("F1",CultureInfo.InvariantCulture)))+" ; elles montent avec n.");

Trois croissances comptées pas à pas 10^1 10^2 10^3 10^4 1 10 100 1,000 10,000 100,000 1,000,000 taille de l’entrée n (échelle log) nombre de pas (échelle log) Tri fusion : n log n Paires : n² Sous-ensembles : 2^n

Pentes log-log : tri fusion 1,20, paires 2,02.


Pentes locales des sous-ensembles (échantillon) : 1.7  3.8  5.9  8.0  10.0  12.1 ; elles montent avec n.


**Lecture.** Sur le graphique, le tri fusion et la paire la plus proche sont des droites, avec les
pentes affichées : un peu plus de 1 pour $n \log n$, 2 pour $n^2$. La courbe des sous-ensembles se
redresse sans cesse — ses pentes locales augmentent d'un point de mesure à l'autre. Aucun polynôme,
même de degré 10 ou 100, ne finit par la suivre : **une exponentielle dépasse tout polynôme**.

On dit qu'un algorithme a un coût **polynomial** si son nombre de pas est au plus de l'ordre de
$n^k$ pour un $k$ fixe ($n$, $n \log n$, $n^2$, $n^3$…), et **exponentiel** s'il est de l'ordre de
$2^n$ ou plus. Cette frontière est la plus importante de toute la série.

### Exercice 3 — Prédire le point de bascule

Un algorithme A coûte $100 \cdot n^2$ pas ; un algorithme B coûte $2^n$ pas. Pour de petites
entrées, B est **moins** cher que A (vérifiez-le pour $n = 5$). À partir de quelle taille $n$ B
devient-il plus cher que A, et le reste-t-il ensuite ?

- **Etape 1 :** prédire la réponse à la main, en calculant quelques valeurs de $100 n^2$ et de $2^n$.
- **Etape 2 :** renseigner `pointBascule` avec le plus petit $n \ge 1$ tel que
  `2^n > 100*n*n` **et** `2^m > 100*m*m` pour tous les $m$ de $n$ à $n + 20$.
- **Indice :** une simple boucle `while` sur $n$ suffit.

In [10]:
int? pointBascule = null; // TODO etudiant : plus petit seuil de dépassement durable pour l'exemple
if(pointBascule is int n)
{
    if(n<1 || n>100)throw new Exception("Exercice 3 : le seuil doit être un entier positif fini dans la plage de vérification.");
    bool durable=Enumerable.Range(n,21).All(m=>BigInteger.Pow(2,m)>100*m*m);
    bool minimal=n==1 || !Enumerable.Range(1,n-1).Any(k=>Enumerable.Range(k,21).All(m=>BigInteger.Pow(2,m)>100*m*m));
    if(!durable || !minimal)throw new Exception("Exercice 3 : le point de bascule n'est pas le plus petit seuil durable vérifié.");
    Console.WriteLine($"Point de bascule vérifié sur la fenêtre demandée : n={n}.");
}
else Console.WriteLine("Exercice à compléter : fournir le plus petit n où 2ⁿ dépasse durablement 100 n².");

Exercice à compléter : fournir le plus petit n où 2ⁿ dépasse durablement 100 n².


## 5. Ce que « croître comme » veut dire en pratique

Les constantes comptent peu face à la forme de la croissance. Pour le voir, supposons une machine
qui exécute $10^8$ pas par seconde (l'ordre de grandeur d'un processeur actuel sur des opérations
simples) et demandons : **quelle est la plus grande entrée traitable** en une seconde, une minute,
un jour ?

La cellule ne cherche pas cette taille en essayant les valeurs de $n$ une à une : elle double
$n$ tant que le coût tient dans le budget, puis resserre l'intervalle par **dichotomie**, la
même idée que la recherche de l'exercice 1. C'est déjà une petite leçon de complexité : pour
des tailles de l'ordre de cent milliards, moins d'une quarantaine de doublements puis autant
d'étapes de dichotomie suffisent, là où l'énumération demanderait cent milliards d'essais.

In [11]:
const long VITESSE = 100_000_000;
var budgetsTemps = new[]{("1 seconde",1L),("1 minute",60L),("1 jour",86_400L)};
var coutsTemps = new (string nom,Func<long,double> cout)[]{
    ("n log2 n",n=>n>1?n*Math.Log2(n):1),
    ("n²",n=>(double)(new BigInteger(n)*n)),
    ("2ⁿ",n=>(double)(BigInteger.One << checked((int)n)))
};
long PlusGrandeTaille(Func<long,double> cout,long budget)
{
    long haut=1;
    while(haut<=long.MaxValue/2 && cout(haut*2)<=budget)haut*=2;
    long bas=haut; haut=haut*2;
    while(haut-bas>1){long milieu=bas+(haut-bas)/2;if(cout(milieu)<=budget)bas=milieu;else haut=milieu;}
    return bas;
}
Console.WriteLine($"{"coût",9} | "+string.Join(" | ",budgetsTemps.Select(x=>$"{x.Item1,15}")));
Console.WriteLine(new string('-',62));
var taillesMax = new Dictionary<string,long[]>();
foreach(var (nom,cout) in coutsTemps)
{
    long[] ligne=budgetsTemps.Select(b=>PlusGrandeTaille(cout,VITESSE*b.Item2)).ToArray();taillesMax[nom]=ligne;
    Console.WriteLine($"{nom,9} | "+string.Join(" | ",ligne.Select(n=>$"{n,15:N0}")));
}
Console.WriteLine();Console.WriteLine("Les tailles maximales sont recherchées par doublement puis dichotomie, sans balayer chaque n.");
foreach(var (nom,ligne) in taillesMax)Console.WriteLine($"{nom,9} : {ligne[0]:N0} à 1 s, {ligne[2]:N0} à 1 jour.");

     coût |       1 seconde |        1 minute |          1 jour


--------------------------------------------------------------


 n log2 n |       4 523 071 |     216 677 120 | 228 957 337 705


       n² |          10 000 |          77 459 |       2 939 387


       2ⁿ |              26 |              32 |              42


Les tailles maximales sont recherchées par doublement puis dichotomie, sans balayer chaque n.


 n log2 n : 4 523 071 à 1 s, 228 957 337 705 à 1 jour.


       n² : 10 000 à 1 s, 2 939 387 à 1 jour.


       2ⁿ : 26 à 1 s, 42 à 1 jour.


**Lecture.** Le tableau résume la leçon du notebook :

- avec un coût $n \log n$ ou $n^2$, multiplier le temps disponible par 86 400 multiplie la taille
  traitable par un grand facteur : environ 50 000 pour le tri (de 4,5 millions à 229 milliards
  d'éléments), environ 294 pour les paires (de 10 000 à près de 3 millions, car
  $\sqrt{86\,400} \approx 294$) ;
- avec un coût $2^n$, le même effort ne fait gagner que **16 éléments** : 86 400 est compris
  entre $2^{16}$ et $2^{17}$, et chaque élément ajouté double le travail. Attendre plus longtemps,
  ou acheter une machine mille fois plus rapide (+10 éléments seulement, car $2^{10} = 1024$), ne
  change presque rien.

C'est pourquoi on appelle l'exponentielle le **mur de l'ingénieur** : on ne le franchit pas avec
du matériel, seulement avec un meilleur **algorithme** — quand il en existe un. Toute la suite de
la série tourne autour de cette question.

## Conclusion

| Idée | Ce que le notebook a mesuré |
|---|---|
| Le coût se compte en **pas**, pas en secondes | chronos variables, nombre de pas identique aux sept essais |
| On étudie le coût en fonction de la **taille** $n$, dans le **pire cas** | recherche linéaire : $T(n) = n$ exactement |
| Une machine de Turing rend le pas **exact** | incrément en $n + k + 2$ pas, où $k$ est la longueur de la traîne de 1 finale ; palindrome en un nombre de pas de l'ordre de $n^2$ |
| Le **test du doublement** reconnaît la croissance | facteur proche de 2 pour $n \log n$, proche de 4 pour $n^2$, ×2 par élément ajouté pour $2^n$ |
| L'exponentielle est un **mur** | un jour au lieu d'une seconde : +16 éléments seulement pour $2^n$ |

*Coût exact de l'incrément.* Un aller de $n + 1$ pas jusqu'au bout du mot, puis un pas par 1 de la
traîne finale, puis un pour poser le chiffre : $n + k + 2$ pas, où $k$ est la longueur de cette
traîne finale de 1. La colonne `pas inc` mesure des mots qui **finissent par 0** ($k = 0$), d'où
$n + 2$ — 10, 18, 34, 66 et 130 pas pour $n$ = 8, 16, 32, 64 et 128. Ce n'est donc pas le fait
d'être un **palindrome** qui donne $n + 2$ : `1001` et `11011` sont des palindromes, finissent par
des 1, et coûtent donc plus que $n + 2$. À l'inverse `1011`, mesuré dans la démonstration de l’incrément, n'est pas un
palindrome et coûte 8 pas pour $n = 4$, soit exactement $n + k + 2$ avec $k = 2$. Le pire cas est
le mot de tous les 1, où la traîne est le mot entier et le coût vaut $2n + 2$ : c'est ce que mesure
`increment.Run("111")`, 8 pas pour $n = 3$. Le coût reste **linéaire** dans tous les régimes — seule la
constante du pire cas change.

On sait désormais **compter** le coût d'un algorithme donné. Mais un même problème admet souvent
plusieurs algorithmes : trier se fait en $n^2$ ou en $n \log n$. Le notebook suivant pose la
question qui structure toute la discipline : pour certains problèmes, **trouver** une solution
semble exiger un coût exponentiel, alors que **vérifier** une solution proposée ne coûte presque
rien. Cet écart est-il inévitable ?

**Suite** : [Complexity-02 — Vérifier ou trouver >>](Complexity-02-P-NP-Reduction-Python.ipynb)

## À retenir

1. **Le chrono varie, le compte de pas reste exact.** Sept parcours de la liste d’un million d’entiers donnent chacun 1 000 000 additions et la même somme. Le temps affiché dépend de l’exécution.
2. **Le pire cas de la recherche linéaire est n, le meilleur est 1.** Pour les tailles de 10 à 100 000, le compteur du pire cas vaut exactement n.
3. **Les deux machines ont des croissances différentes.** L’incrément coûte n+k+2 transitions, où k est la longueur de la traîne finale de 1. Le palindrome mesuré coûte 12 609 transitions à n=128; son quotient par n² se rapproche de 0,77 sur les tailles montrées.
4. **Le test du doublement distingue les trois profils.** Le tri fusion passe de 93 625 à 203 252 comparaisons entre 8 000 et 16 000 entrées. Pour les paires, le rapport suit $4+2/(n-1)$ (il est affiché 4,00 après arrondi aux grandes tailles). Le nombre de sous-ensembles examinés double à chaque élément ajouté.
5. **Le budget se traduit en taille traitable.** À $10^8$ pas/s, le modèle $n²$ passe de 10 000 éléments en une seconde à 2 939 387 en un jour; le modèle $2^n$ passe de 26 à 42.


### Les trois exercices restent à résoudre

| Exercice | Travail demandé | Contrôle du notebook |
|---|---|---|
| 1 — recherche dichotomique | Retourner l’indice et compter les comparaisons sur des listes triées. | Cas trouvé, liste vide, absence sous le minimum et au-dessus du maximum; compte exact pour $2^k$. |
| 2 — machine de parité | Construire les transitions qui acceptent les mots ayant un nombre pair de `1`. | 511 mots binaires courts contre un oracle de parité, puis longueurs 10, 100 et 1 000. |
| 3 — point de bascule | Trouver le premier seuil durable entre $100n^2$ et $2^n$. | Le seuil doit être minimal et le dépassement tenir sur la fenêtre testée. |


## Vérifiez votre compréhension

1. Les sept chronos portent chacun sur un million d’additions. Qu'est-ce qui varie, qu'est-ce qui ne varie pas, et lequel des deux la théorie de la complexité doit-elle suivre ?
2. Le palindrome coûte 96,99× l'incrément à $n = 128$, mais le rapport palindrome / $n^2$ reste autour de 0,77. Pourquoi le second rapport renseigne-t-il mieux sur le degré de la croissance que le premier ?
3. Le tri fusion double presque son coût (×2,17) quand $n$ double; les paires affichent 4,00 après arrondi, alors que leur facteur exact est $4+2/(n-1)$ et tend vers 4. Que donne le test de doublement sur les sous-ensembles, et qu'est-ce que cela dit de leur croissance ?
4. À $10^8$ pas/s, $n^2$ tient 10 000 éléments en une seconde et $2^n$ seulement 26. Le passage d'une seconde à un jour est-il de même nature pour les deux croissances ?

<details><summary>Réponses</summary>

1. Le chrono dépend de la machine (caches, interruptions, fréquence effective) ; le compte de pas ne dépend que de l'algorithme et de l'entrée. La complexité mesure le second : c'est le seul des deux qui se retrouve sur une autre machine.
2. Le rapport à $n^2$ se stabilise quand $n$ double (1,078 puis 0,910, 0,829, 0,789, 0,770) : une constante à l'infini est la signature d'une croissance en $n^2$. Le rapport à l'incrément, lui, croît avec $n$ (6,90× puis 12,94×, 24,97×, 48,98×, 96,99×) : il mélange la croissance du palindrome avec celle de l'incrément et ne dit rien sur le degré.
3. Rien de constant : quand $n$ double, le coût est multiplié par 32 puis par 1 024 ; quand $n$ augmente de 1, il est multiplié par 2. C'est la signature de $2^n$ — le carnet le vérifie dans les deux sens, doublement d'une part, incrément de 1 d'autre part.
4. Non. Pour $2^n$, ×86 400 ne donne que +16 sur $n$ (26 → 42) : le facteur constant devant l'exposant ne se retrouve pas dans l'exposant. Pour $n^2$, la taille traitable passe de 10 000 à 2 939 387, soit un facteur ≈294 — qui est précisément $\sqrt{86\,400}$ : un gain de temps constant se retrouve à la racine quand la croissance est polynomiale, et se perd presque quand elle est exponentielle.

</details>

## Pour aller plus loin

- **Carnet suivant** : [Complexity-02 — Vérifier ou trouver : P, NP et la réduction](Complexity-02-P-NP-Reduction-Python.ipynb) — le pas se compte, mais sur quelle grandeur : le nombre d'appels du solveur, ou le nombre de lectures du vérificateur ?
- **Dans la même série** : [Complexity-03 — Plus de temps, plus de problèmes](Complexity-03-HierarchieTemporelle-Python.ipynb) — le budget devient une hiérarchie de classes ; [Complexity-03b — Hartmanis–Stearns](Complexity-03b-HartmanisStearns-HierarchieTemporelle-Python.ipynb) — le compteur de pas poussé jusqu'au théorème de 1965.
- **Carnet voisin d'une autre série** : [Sudoku-01 — Backtracking (Python)](../Sudoku/Sudoku-01-Backtracking-Python.ipynb) — le retour arrière, premier usage du budget : à chaque essai on repart d'un chemin, et le coût se compte en nœuds explorés.
- **Lectures** : Impagliazzo et Paturi (2001), *On the Complexity of k-SAT*, J. Comput. Syst. Sci. 62-2 — la mesure du coût $n \cdot 2^n$ que ce carnet compte à la main (copie archivée au gisement partagé, dossier `Bibliographie IA/Automata/`). La source historique du geste — Hartmanis et Stearns (1965) — est reprise et référencée dans [03b](Complexity-03b-HartmanisStearns-HierarchieTemporelle-Python.ipynb).